# 02 Item2Vec Baseline: Books + Movies Intersection

## Setup

In [ ]:
!git clone https://github.com/mkobycheva/recommendation-system.git 2>/dev/null \
  || (cd recommendation-system && git pull)

%cd recommendation-system
!git checkout item-2-vec && git pull
!pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
from gensim.models import Word2Vec

from configs.model_configs import ITEM2VEC_CONFIG
from src.data.build_matrix import add_domain_item_ids
from src.evaluation.cross_domain_eval import evaluate_multi_domain


## Load Prepared Splits

In [ ]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [ ]:
books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()


## Data prep for item2vec

In [ ]:
def build_item_sequences(interactions_df):
    return (
        interactions_df.sort_values('timestamp')
        .groupby('user_id')['item_id']
        .apply(list)
        .tolist()
    )


sequences = build_item_sequences(train_df)

print(f'Total sequences: {len(sequences)}')
print(f'Example: {sequences[0][:5]}')


## Word2Vec applied to Multi-Domain reommendation problem

In [ ]:
def train_item2vec(sequences):
    return Word2Vec(
        sentences=sequences,
        vector_size=ITEM2VEC_CONFIG['vector_size'],
        window=ITEM2VEC_CONFIG['window'],
        sg=ITEM2VEC_CONFIG['sg'],
        min_count=ITEM2VEC_CONFIG['min_count'],
        workers=ITEM2VEC_CONFIG['workers'],
        seed=ITEM2VEC_CONFIG['seed'],
    )


model = train_item2vec(sequences)

print(f'Vocabulary size: {len(model.wv)}')
print(f'Example vector shape: {model.wv[sequences[0][0]].shape}')


# Recommendations

In [ ]:
def domain_items_for_model(trained_model, domains=('books', 'movies')):
    return {
        domain: np.array([
            item for item in trained_model.wv.index_to_key
            if item.startswith(f'{domain}::')
        ])
        for domain in domains
    }


domain_items = domain_items_for_model(model)

print(f"Books in vocab:  {len(domain_items['books']):,}")
print(f"Movies in vocab: {len(domain_items['movies']):,}")


In [ ]:
def user_items_by_train_history(train_interactions_df):
    return (
        train_interactions_df.groupby('user_id')['item_id']
        .apply(list)
        .to_dict()
    )


user_train_items = user_items_by_train_history(train_df)


In [ ]:
def make_item2vec_recommender(trained_model, train_items_by_user, domains=('books', 'movies')):
    model_domain_items = domain_items_for_model(trained_model, domains=domains)

    def recommend_for_users(user_ids, target_domain, k=10, batch_size=None):
        if batch_size is None:
            batch_size = ITEM2VEC_CONFIG['batch_size']

        candidates = model_domain_items.get(target_domain, np.array([], dtype=object))
        recommendations = {}
        user_ids = list(user_ids)

        if len(candidates) == 0:
            return {user_id: [] for user_id in user_ids}

        candidate_vectors = trained_model.wv[candidates]
        candidate_to_idx = {item: i for i, item in enumerate(candidates)}

        for batch_start in range(0, len(user_ids), batch_size):
            batch = user_ids[batch_start:batch_start + batch_size]
            user_vectors = []
            valid_user_ids = []
            known_items_list = []

            for user_id in batch:
                user_items = train_items_by_user.get(user_id, [])
                valid_items = [item for item in user_items if item in trained_model.wv]
                if not valid_items:
                    recommendations[user_id] = []
                    continue
                user_vectors.append(np.mean(trained_model.wv[valid_items], axis=0))
                valid_user_ids.append(user_id)
                known_items_list.append(set(user_items))

            if not user_vectors:
                continue

            user_matrix = np.stack(user_vectors)
            all_scores = user_matrix @ candidate_vectors.T

            for i, user_id in enumerate(valid_user_ids):
                scores = all_scores[i].copy()
                known_indices = np.array([
                    candidate_to_idx[item]
                    for item in known_items_list[i]
                    if item in candidate_to_idx
                ], dtype=np.int64)
                if len(known_indices) > 0:
                    scores[known_indices] = -np.inf

                finite_count = np.isfinite(scores).sum()
                if finite_count == 0:
                    recommendations[user_id] = []
                    continue

                top_n = min(k, int(finite_count))
                partition_k = min(top_n - 1, len(scores) - 1)
                top_indices = np.argpartition(-scores, partition_k)[:top_n]
                top_indices = top_indices[np.argsort(-scores[top_indices])]
                recommendations[user_id] = [candidates[j] for j in top_indices]

            if batch_start % 10000 == 0:
                print(f"Processed {batch_start}/{len(user_ids)} users...")

        return recommendations

    return recommend_for_users


recommend_for_users = make_item2vec_recommender(model, user_train_items)


## Evaluate and Save Metrics

In [ ]:
def item2vec_result_row(model_name, train_interactions_df, valid_results, test_results):
    row = {
        'model': model_name,
        'vector_size': ITEM2VEC_CONFIG['vector_size'],
        'window': ITEM2VEC_CONFIG['window'],
        'sg': ITEM2VEC_CONFIG['sg'],
        'min_count': ITEM2VEC_CONFIG['min_count'],
        'k': ITEM2VEC_CONFIG['k'],
        'n_users': train_interactions_df['user_id'].nunique(),
        'n_items': train_interactions_df['item_id'].nunique(),
        'n_train_interactions': len(train_interactions_df),
    }

    for split_name, results in [('valid', valid_results), ('test', test_results)]:
        for metric in ['ndcg@10', 'recall@10', 'map@5']:
            values = []
            for domain in ['books', 'movies']:
                value = results.get(domain, {}).get(metric, np.nan)
                row[f'{domain}_{split_name}_{metric}'] = value
                if not pd.isna(value):
                    values.append(value)
            row[f'{split_name}_macro_{metric}'] = round(float(np.mean(values)), 4) if values else np.nan

    return row


In [ ]:
K = ITEM2VEC_CONFIG['k']

joint_valid_results, joint_test_results = evaluate_multi_domain(
    split_dfs={'valid': valid_df, 'test': test_df},
    recommend_func=recommend_for_users,
    k=K,
    map_k=5,
)

joint_result_row = item2vec_result_row(
    'item2vec_books_movies_joint',
    train_df,
    joint_valid_results,
    joint_test_results,
)

joint_result_row


## Single-Domain Ablation


In [ ]:
books_only_sequences = build_item_sequences(books_train)
books_only_model = train_item2vec(books_only_sequences)
books_only_recommend_for_users = make_item2vec_recommender(
    books_only_model,
    user_items_by_train_history(books_train),
    domains=('books',),
)

books_only_valid_results, books_only_test_results = evaluate_multi_domain(
    split_dfs={
        'valid': valid_df[valid_df['domain'] == 'books'],
        'test': test_df[test_df['domain'] == 'books'],
    },
    recommend_func=books_only_recommend_for_users,
    k=K,
    map_k=5,
)

books_only_result_row = item2vec_result_row(
    'item2vec_books_only',
    books_train,
    books_only_valid_results,
    books_only_test_results,
)

books_only_result_row


In [ ]:
movies_only_sequences = build_item_sequences(movies_train)
movies_only_model = train_item2vec(movies_only_sequences)
movies_only_recommend_for_users = make_item2vec_recommender(
    movies_only_model,
    user_items_by_train_history(movies_train),
    domains=('movies',),
)

movies_only_valid_results, movies_only_test_results = evaluate_multi_domain(
    split_dfs={
        'valid': valid_df[valid_df['domain'] == 'movies'],
        'test': test_df[test_df['domain'] == 'movies'],
    },
    recommend_func=movies_only_recommend_for_users,
    k=K,
    map_k=5,
)

movies_only_result_row = item2vec_result_row(
    'item2vec_movies_only',
    movies_train,
    movies_only_valid_results,
    movies_only_test_results,
)

movies_only_result_row


## Baselines: Top-Popular and Random


In [ ]:
from src.data.build_matrix import build_user_item_matrix
from src.evaluation.baselines import make_popularity_recommender, make_random_recommender

interaction_matrix_for_baselines = build_user_item_matrix(train_df)
domain_item_indices = interaction_matrix_for_baselines.domain_item_indices
idx2item = interaction_matrix_for_baselines.idx2item

popularity_recommend_func = make_popularity_recommender(train_df, domain_item_indices, idx2item)
random_recommend_func = make_random_recommender(domain_item_indices, idx2item, seed=42)

popularity_valid_results, popularity_test_results = evaluate_multi_domain(
    split_dfs={'valid': valid_df, 'test': test_df},
    recommend_func=popularity_recommend_func,
    k=K,
    map_k=5,
)

random_valid_results, random_test_results = evaluate_multi_domain(
    split_dfs={'valid': valid_df, 'test': test_df},
    recommend_func=random_recommend_func,
    k=K,
    map_k=5,
)


def baseline_result_row(model_name, valid_results, test_results):
    row = {
        'model': model_name,
        'vector_size': np.nan,
        'window': np.nan,
        'sg': np.nan,
        'min_count': np.nan,
        'k': K,
        'n_users': train_df['user_id'].nunique(),
        'n_items': train_df['item_id'].nunique(),
        'n_train_interactions': len(train_df),
    }

    for split_name, results in [('valid', valid_results), ('test', test_results)]:
        for metric in ['ndcg@10', 'recall@10', 'map@5']:
            values = []
            for domain in ['books', 'movies']:
                value = results.get(domain, {}).get(metric, np.nan)
                row[f'{domain}_{split_name}_{metric}'] = value
                if not pd.isna(value):
                    values.append(value)
            row[f'{split_name}_macro_{metric}'] = round(float(np.mean(values)), 4) if values else np.nan

    return row


popularity_result_row = baseline_result_row('popularity_baseline', popularity_valid_results, popularity_test_results)
random_result_row = baseline_result_row('random_baseline', random_valid_results, random_test_results)


In [ ]:
from src.evaluation.lift import compute_lift_table

print("=== Books (test) ===")
print(compute_lift_table(joint_test_results['books'], popularity_test_results['books'], random_test_results['books']))

print("\n=== Movies (test) ===")
print(compute_lift_table(joint_test_results['movies'], popularity_test_results['movies'], random_test_results['movies']))


In [ ]:
item2vec_ablation_results = pd.DataFrame([
    books_only_result_row,
    movies_only_result_row,
    joint_result_row,
    popularity_result_row,
    random_result_row,
])

os.makedirs('results', exist_ok=True)
item2vec_ablation_results.to_csv('results/item2vec_ablation_metrics.csv', index=False)
item2vec_ablation_results
